In [1]:
import os
import pandas as pd

In [35]:
receptor_transmitter_file = "data/receptor_transmitter_map.csv"
rec_trans_df = pd.read_csv(receptor_transmitter_file)

neuron_to_receptor_file = "data/neuron_to_receptors.csv"
neur_rec_df = pd.read_csv(neuron_to_receptor_file)

neuron_to_transmitter_file = "data/neuron_to_neurotransmitters.csv"
neur_trans_df = pd.read_csv(neuron_to_transmitter_file)

connectome_file = "data/herm_full_edgelist.csv"
connectome_df = pd.read_csv(connectome_file)

In [48]:
chem_connectome_df = connectome_df[connectome_df['Type'] == "chemical"].copy()
chem_connectome_df['Source'] = chem_connectome_df['Source'].apply(lambda x: x.strip())
chem_connectome_df['Target'] = chem_connectome_df['Target'].apply(lambda x: x.strip())

In [49]:
chem_connectome_df

,Source,Target,Weight,Type
0,I1L,I2L,10,chemical
1,I1L,I3,3,chemical
2,I1L,I5,2,chemical
3,I1L,I6,1,chemical
4,I1L,M2L,3,chemical
...,...,...,...,...
4676,VC06,DD06,2,chemical
4677,VC06,VD11,5,chemical
4678,VC06,VD12,3,chemical
4679,VC06,vBWML17,1,chemical


In [50]:
print(len(neur_rec_df['neuron'].unique()))
print(len(neur_trans_df['neuron'].unique()))
print(len(chem_connectome_df['Source'].unique()))
print(len(chem_connectome_df['Target'].unique()))

print(chem_connectome_df["Source"].isin(neur_trans_df['neuron'].unique()))

279
243
298
418
0        True
1        True
2        True
3        True
4        True
        ...  
4676    False
4677    False
4678    False
4679    False
4680    False
Name: Source, Length: 4681, dtype: bool


In [59]:
chem_connectome_neurons_df = chem_connectome_df[chem_connectome_df['Source'].isin(neur_trans_df['neuron']) & chem_connectome_df['Target'].isin(neur_rec_df['neuron'])]

In [60]:
chem_connectome_neurons_df

,Source,Target,Weight,Type
0,I1L,I2L,10,chemical
1,I1L,I3,3,chemical
3,I1L,I6,1,chemical
4,I1L,M2L,3,chemical
5,I1L,M3L,8,chemical
...,...,...,...,...
4549,HSNR,RIMR,1,chemical
4550,HSNR,RMDDR,1,chemical
4551,HSNR,SABD,1,chemical
4552,HSNR,SABVR,1,chemical


In [61]:
chem_connectome_neurons_df['PreSynTransmitter'] = chem_connectome_neurons_df['Source'].apply(
    lambda x: ", ".join(neur_trans_df.loc[neur_trans_df['neuron'] == x, 'neurotransmitter'].tolist())
    if not neur_trans_df.loc[neur_trans_df['neuron'] == x, 'neurotransmitter'].empty else None
)

/tmp/ipykernel_92378/1440854830.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chem_connectome_neurons_df['PreSynTransmitter'] = chem_connectome_neurons_df['Source'].apply(


In [62]:
chem_connectome_neurons_df

,Source,Target,Weight,Type,PreSynTransmitter
0,I1L,I2L,10,chemical,Acetylcholine
1,I1L,I3,3,chemical,Acetylcholine
3,I1L,I6,1,chemical,Acetylcholine
4,I1L,M2L,3,chemical,Acetylcholine
5,I1L,M3L,8,chemical,Acetylcholine
...,...,...,...,...,...
4549,HSNR,RIMR,1,chemical,Serotonin
4550,HSNR,RMDDR,1,chemical,Serotonin
4551,HSNR,SABD,1,chemical,Serotonin
4552,HSNR,SABVR,1,chemical,Serotonin


In [63]:
def get_postsyn_receptor(target, presyn_transmitter):
    # get all receptors associated with the target neuron from neur_rec_df
    target_receptors = neur_rec_df[neur_rec_df['neuron'] == target]['receptor'].tolist()
    # split transmitters by comma in case there are more than one
    transmitter_list = [t.strip() for t in presyn_transmitter.split(',')]
    # check each receptor to see if its transmitter matches any in transmitter_list using rec_trans_df
    for receptor in target_receptors:
        match = rec_trans_df[rec_trans_df['receptor'] == receptor]
        if not match.empty and match.iloc[0]['transmitter'] in transmitter_list:
            return receptor
    return None

chem_connectome_neurons_df['PostSynReceptor'] = chem_connectome_neurons_df.apply(
    lambda row: get_postsyn_receptor(row['Target'], row['PreSynTransmitter']), axis=1)

/tmp/ipykernel_92378/3756589743.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chem_connectome_neurons_df['PostSynReceptor'] = chem_connectome_neurons_df.apply(


In [ ]:
chem_connectome_neurons_df.to_csv("data/chemical_connectome.csv", index=False, sep=';')
chem_connectome_neurons_df

,Source,Target,Weight,Type,PreSynTransmitter,PostSynReceptor
0,I1L,I2L,10,chemical,Acetylcholine,None
1,I1L,I3,3,chemical,Acetylcholine,None
3,I1L,I6,1,chemical,Acetylcholine,None
4,I1L,M2L,3,chemical,Acetylcholine,None
5,I1L,M3L,8,chemical,Acetylcholine,None
...,...,...,...,...,...,...
4549,HSNR,RIMR,1,chemical,Serotonin,MOD-1
4550,HSNR,RMDDR,1,chemical,Serotonin,SER-1
4551,HSNR,SABD,1,chemical,Serotonin,SER-2
4552,HSNR,SABVR,1,chemical,Serotonin,SER-2
